# ML classique : ce que le syllabus IOAI demande de maitriser

Avant les reseaux de neurones, le syllabus (theorie ET pratique) attend une maitrise solide de :
- regression lineaire / logistique
- kNN, arbres de decision, ensemble learning
- bias-variance tradeoff, cross-validation
- clustering (k-means), PCA

Ce notebook va droit au but : chaque methode = definition courte + code qui tourne + piege classique a eviter.

In [ ]:
import numpy as np
import torch
np.random.seed(0)

## 1. Regression lineaire

Predire une valeur continue `y` a partir de features `X`. Le modele apprend des poids `w` et un biais `b` tels que `y_pred = X @ w + b` minimise l'erreur quadratique.

**Piege classique** : oublier de normaliser les features avant l'entrainement, ce qui ralentit ou empeche la convergence.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# donnees synthetiques
N = 200
X = np.random.randn(N, 3)
vrais_poids = np.array([2.0, -1.0, 0.5])
y = X @ vrais_poids + 3.0 + np.random.randn(N) * 0.1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

modele = LinearRegression()
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print("poids appris:", modele.coef_)
print("biais appris:", modele.intercept_)
print("MSE sur test:", mean_squared_error(y_test, y_pred))

## 2. Regression logistique (classification binaire)

Meme principe que la regression lineaire mais on passe la sortie dans une sigmoide pour obtenir une probabilite entre 0 et 1.

**Memotech** : "logistique = lineaire + sigmoide". La frontiere de decision reste lineaire, seule la sortie devient une probabilite.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = np.random.randn(N, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

modele = LogisticRegression()
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print("accuracy:", accuracy_score(y_test, y_pred))

## 3. kNN, arbres de decision, ensemble learning

- **kNN** : classe un point selon la majorite de ses k plus proches voisins. Pas d'entrainement a proprement parler, tout se joue au moment de la prediction (lazy learning).
- **Arbre de decision** : suite de tests sur les features (`si feature_2 > 0.5 alors ...`). Facile a interpreter mais overfit vite si pas limite en profondeur.
- **Ensemble learning** : combiner plusieurs modeles faibles pour un modele fort. Deux familles a connaitre :
  - **Bagging** (ex: Random Forest) : plusieurs arbres entraines sur des sous-echantillons differents, on moyenne leurs votes -> reduit la variance
  - **Boosting** (ex: Gradient Boosting) : les modeles sont entraines en sequence, chacun corrige les erreurs du precedent -> reduit le biais

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

for nom, modele in [
    ("kNN (k=5)", KNeighborsClassifier(n_neighbors=5)),
    ("Arbre de decision (profondeur max 4)", DecisionTreeClassifier(max_depth=4, random_state=0)),
    ("Random Forest (50 arbres)", RandomForestClassifier(n_estimators=50, random_state=0)),
]:
    modele.fit(X_train, y_train)
    acc = accuracy_score(y_test, modele.predict(X_test))
    print(f"{nom:40s} accuracy = {acc:.3f}")

## 4. Bias-variance tradeoff : le concept le plus souvent teste en theorie

- **Biais eleve** (underfitting) : le modele est trop simple, il rate des patterns meme dans les donnees d'entrainement. Erreur train ET test elevees.
- **Variance elevee** (overfitting) : le modele colle trop aux donnees d'entrainement, y compris au bruit. Erreur train faible, erreur test elevee.

**Memotech** : "biais = pas assez appris, variance = trop appris par coeur".

Demonstration concrete : on fait varier la profondeur d'un arbre de decision et on observe l'ecart train/test.

In [ ]:
profondeurs = [1, 2, 3, 5, 8, 12, None]
print(f"{'profondeur':12s} {'acc train':10s} {'acc test':10s}")
for prof in profondeurs:
    modele = DecisionTreeClassifier(max_depth=prof, random_state=0)
    modele.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, modele.predict(X_train))
    acc_test = accuracy_score(y_test, modele.predict(X_test))
    print(f"{str(prof):12s} {acc_train:10.3f} {acc_test:10.3f}")

# a observer : profondeur faible -> les deux accuracies sont mediocres (biais)
# profondeur elevee/None -> acc train proche de 1.0 mais acc test qui stagne ou baisse (variance)

## 5. Cross-validation

Une seule separation train/test peut donner un score qui depend du hasard du split. La k-fold cross-validation decoupe les donnees en k blocs, entraine k fois en gardant a chaque fois un bloc different pour le test, et moyenne les scores.

**Regle pratique** : k=5 ou k=10 sont les valeurs standards. Toujours reporter la moyenne ET l'ecart-type des scores, pas juste la moyenne.

In [ ]:
from sklearn.model_selection import cross_val_score

modele = RandomForestClassifier(n_estimators=50, random_state=0)
scores = cross_val_score(modele, X, y, cv=5)
print("scores des 5 folds:", scores)
print(f"moyenne = {scores.mean():.3f}, ecart-type = {scores.std():.3f}")

## 6. Clustering non supervise : k-means

Pas de labels ici, on cherche a regrouper les points en k clusters. L'algorithme alterne :
1. assigner chaque point au centre le plus proche
2. recalculer chaque centre comme la moyenne des points qui lui sont assignes

jusqu'a convergence.

**Piege classique** : k-means a besoin de connaitre k a l'avance, et le resultat depend de l'initialisation (relancer plusieurs fois, `n_init` dans sklearn).

In [ ]:
from sklearn.cluster import KMeans

# 3 blobs synthetiques
centres_vrais = np.array([[0, 0], [5, 5], [0, 5]])
X_clusters = np.vstack([c + np.random.randn(50, 2) * 0.5 for c in centres_vrais])

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(X_clusters)

print("centres trouves:")
print(kmeans.cluster_centers_)
print("nb de points par cluster:", np.bincount(labels))

## 7. PCA (reduction de dimension)

PCA projette les donnees sur les axes qui capturent le plus de variance. Utile pour visualiser des donnees en haute dimension ou pour reduire le bruit avant un autre modele.

**Memotech** : "PCA = trouver les meilleurs angles de vue pour ne rien perdre d'important".

In [ ]:
from sklearn.decomposition import PCA

# donnees a 5 dimensions, mais en realite generees a partir de 2 facteurs caches
facteurs_caches = np.random.randn(100, 2)
projection = np.random.randn(2, 5)
X_haute_dim = facteurs_caches @ projection + np.random.randn(100, 5) * 0.05

pca = PCA(n_components=2)
X_reduit = pca.fit_transform(X_haute_dim)

print("variance expliquee par chaque composante:", pca.explained_variance_ratio_)
print("shape apres reduction:", X_reduit.shape)   # (100, 2) au lieu de (100, 5)

## Exercice recapitulatif (10-15 min)

1. Genere un dataset de classification a 2 classes avec du bruit (`X, y` comme dans la section 2, mais ajoute plus de bruit)
2. Compare kNN, arbre de decision et Random Forest avec une cross-validation a 5 folds
3. Identifie lequel a le meilleur compromis biais-variance (meilleure moyenne, plus petit ecart-type)
4. Bonus : applique une PCA a 2 composantes sur `X` avant de reentrainer, est-ce que ca degrade la performance ?

In [ ]:
# ecris ta solution ici



